In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
fake_data = pd.read_csv('/content/drive/MyDrive/ML/true_fake_news_classification/Fake.csv')
true_data = pd.read_csv('/content/drive/MyDrive/ML/true_fake_news_classification/True.csv')

In [ ]:
fake_data["T/F"] = 0
fake_data

In [ ]:
true_data["T/F"] = 1
true_data

In [ ]:
data = pd.concat([fake_data, true_data])

In [ ]:
data

In [ ]:
data = data.sample(frac=1).reset_index(drop=True)
data

In [ ]:
data.drop(columns=["subject", "date"], inplace=True)
data

In [ ]:
data['text'] = data['title'] + '. ' + data['text']
data.drop(columns=['title'], inplace=True)

In [ ]:
data

In [ ]:
max_len = len(data['text'][0])
j = 0;
for i in range(len(data['text'])):
  if max_len < len(data['text'][i]):
    max_len = len(data['text'][i])
    j = i
print(max_len, j)

In [ ]:
len(data['text'][40000])

In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LENGTH = 512
BATCH_SIZE = 8

In [ ]:
def create_chunks(text, label, tokenizer, max_length):
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []
    chunk_size = max_length - 2 # Account for [CLS] and [SEP] tokens

    for i in range(0, len(token_ids), chunk_size):
        chunk = token_ids[i:i + chunk_size]

        input_ids = (
            [tokenizer.cls_token_id]
            + chunk
            + [tokenizer.sep_token_id]
        )

        attention_mask = [1] * len(input_ids)

        padding_length = max_length - len(input_ids)
        input_ids += ([tokenizer.pad_token_id] * padding_length)
        attention_mask += ([0] * padding_length)

        chunks.append(
            {
                'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
                'label': torch.tensor(label, dtype=torch.float)
            }
        )
    return chunks

In [ ]:
def dataframe_to_chunks(df, tokenizer, max_length):
    all_chunks = []
    for _, row in df.iterrows():
        chunks = create_chunks(row["text"], row["T/F"], tokenizer, max_length)
        all_chunks.extend(chunks)
    return all_chunks

In [ ]:
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Creating training chunks...")
train_chunks = dataframe_to_chunks(train_df, tokenizer, MAX_LENGTH)

print("Creating validation chunks...")
val_chunks = dataframe_to_chunks(val_df, tokenizer, MAX_LENGTH)

print("Creating test chunks...")
test_chunks = dataframe_to_chunks(test_df, tokenizer, MAX_LENGTH)

print("\nNumber of chunks:")
print("Train:", len(train_chunks))
print("Validation:", len(val_chunks))
print("Test:", len(test_chunks))

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]

In [ ]:
train_dataset = NewsDataset(train_chunks)
val_dataset = NewsDataset(val_chunks)
test_dataset = NewsDataset(test_chunks)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Verify a batch from the DataLoader
for batch in train_dataloader:
    print("\nInput IDs shape:", batch['input_ids'].shape)
    print("Attention Mask shape:", batch['attention_mask'].shape)
    print("Labels shape:", batch['label'].shape)
    break

### Define the PyTorch Model

In [ ]:
class NewsClassifier(nn.Module):
    def __init__(self, num_classes=1):
        super(NewsClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # Use pooled output for classification
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NewsClassifier(num_classes=1).to(device)
print(model)

### Define Loss Function, Optimizer, and Training Parameters

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
NUM_EPOCHS = 1

### Training Loop

In [ ]:
def train_model(model, dataloader, criterion, optimizer, device, log_interval=100):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Training Batches")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if (batch_idx + 1) % log_interval == 0:
            print(f"  Batch {batch_idx+1}/{len(dataloader)}, Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluation Batches"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs.squeeze(), labels)
            total_loss += loss.item()

            preds = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
            correct_predictions += (preds == labels).sum().item()

    return total_loss / len(dataloader), correct_predictions / (len(dataloader.dataset)) * BATCH_SIZE


print("Starting training...")
for epoch in tqdm(range(NUM_EPOCHS), desc="Epochs"):
    train_loss = train_model(model, train_dataloader, criterion, optimizer, device, log_interval=100)
    val_loss, val_accuracy = evaluate_model(model, val_dataloader, criterion, device)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Training complete.")

### Evaluate Model on Test Set

In [ ]:
test_loss, test_accuracy = evaluate_model(model, test_dataloader, criterion, device)
print(f"\nTest Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")